# CMP701: Computer Programming I
## Week 13 – Error Handling and Debugging (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sakinat-Folorunso/cmp701-python-pgd/blob/main/notebooks/CMP701_Week13_Error_Handling_and_Debugging.ipynb)

**Programme:** PGD Computer Science
**Course Unit:** 3 (LH: 30; PH: 45)
**Platform:** Google Colab (Python 3.10+)

**Prepared by:**
**Dr. Sakinat Folorunso**
Associate Professor of AI Systems
Department of Computer Sciences
Olabisi Onabanjo University, Ago-Iwoye, Nigeria
📧 **Email:** sakinat.folorunso@oouagoiwoye.edu.ng

---

### 🎯 Learning Objectives
By the end of this lab, students should be able to:
- Distinguish syntax, runtime and logical errors
- Read a Python traceback and locate the faulty line
- Use `try / except / else / finally` correctly
- Raise exceptions deliberately and write input-validation functions
- Apply debugging techniques and write simple `assert` tests

## 🧠 Concept Recap
Errors are normal. A good programmer is not one who never makes errors, but one who can **detect, understand and fix** them —
and who writes programs that **fail gracefully**.

## 🧪 Lab 13.1: Three Kinds of Error

**Syntax error** — the grammar is wrong, so the program never starts. (We use `compile()` so this notebook does not stop.)

In [ ]:
source = "x = 7\nif x > 5\n    print(x)"       # missing colon after the condition
try:
    compile(source, "<demo>", "exec")
except SyntaxError as err:
    print("SyntaxError:", err.msg, "| line", err.lineno)

**Runtime error** — the program starts, then fails on a specific operation.

In [ ]:
for risky in (lambda: 10 / 0, lambda: int("abc"), lambda: [1, 2, 3][5], lambda: {"a": 1}["b"], lambda: "2" + 2):
    try:
        risky()
    except Exception as err:
        print(f"{type(err).__name__:<18} {err}")

**Logical error** — the program runs but gives the **wrong answer**. Python cannot detect these; *you* must test.

In [ ]:
a, b = 10, 20
avg_wrong = a + b / 2        # precedence bug: only b is divided
avg_right = (a + b) / 2
print(avg_wrong, avg_right)

## 🧪 Lab 13.2: Reading a Traceback
Read a traceback **from the bottom up**: the last line names the error; the lines above show the chain of calls that led to it.

In [ ]:
import traceback

def average(values):
    return sum(values) / len(values)

def report(values):
    return f"Average = {average(values)}"

try:
    print(report([]))             # empty list -> ZeroDivisionError inside average()
except ZeroDivisionError:
    traceback.print_exc()

## 🧪 Lab 13.3: `try` / `except` and Multiple Exceptions

In [ ]:
def safe_divide(text_a, text_b):
    try:
        a = int(text_a)
        b = int(text_b)
        return a / b
    except ValueError:
        return "Invalid number"
    except ZeroDivisionError:
        return "Cannot divide by zero"

for pair in [("10", "4"), ("10", "0"), ("ten", "2")]:
    print(pair, "->", safe_divide(*pair))

## 🧪 Lab 13.4: `else` and `finally`
- `else` runs only if **no** exception occurred.
- `finally` **always** runs — ideal for cleanup.

In [ ]:
def read_first_line(filename):
    try:
        f = open(filename, "r", encoding="utf-8")
    except FileNotFoundError:
        print("File not found")
    else:
        with f:
            print("First line:", f.readline().strip())
    finally:
        print("Operation completed")

with open("demo13.txt", "w", encoding="utf-8") as f:
    f.write("Hello from Week 13\n")

read_first_line("demo13.txt")
read_first_line("missing.txt")

## 🧪 Lab 13.5: Raising Exceptions and Validating Input

In [ ]:
def parse_score(text):
    """Convert text to a score in 0-100, or raise ValueError with a clear message."""
    try:
        value = float(text)
    except ValueError:
        raise ValueError(f"'{text}' is not a number") from None
    if not 0 <= value <= 100:
        raise ValueError(f"{value} is outside the range 0-100")
    return value

for text in ["75", "100", "abc", "101", "-3"]:
    try:
        print(text, "->", parse_score(text))
    except ValueError as err:
        print(text, "-> rejected:", err)

Now combine it with the **input-validation loop** from Week 6. This is the pattern your mini-project needs.

In [ ]:
def read_score(prompt="Enter score (0-100): "):
    while True:
        try:
            return parse_score(input(prompt))
        except ValueError as err:
            print("Invalid input:", err)

score = read_score()
print("Accepted:", score)

## 🧪 Lab 13.6: Debugging Techniques
1. **Read the error message** — name, message, line number.
2. **Reproduce** the bug with the smallest possible input.
3. **Print debugging** — show variable values at key points.
4. **Trace by hand** with a dry-run table.
5. **Isolate** — comment out or simplify until the bug disappears, then add back.
6. **Explain the code aloud** (rubber-duck debugging).
7. Use a debugger (`breakpoint()` / VS Code) when prints are not enough.

**Worked example.** This function should return the highest score. Find and fix the bug.

In [ ]:
def highest(scores):
    best = 0
    for s in scores:
        if s > best:
            best = s
    return best

print(highest([40, 55, 62]))     # looks fine
print(highest([-5, -2, -9]))     # prints 0 — the correct answer is -2. Why?

In [ ]:
def highest_fixed(scores):
    best = scores[0]             # start from a real element, not an arbitrary 0
    for s in scores:
        if s > best:
            best = s
    return best

print(highest_fixed([-5, -2, -9]))

## 🧪 Lab 13.7: Tiny Tests with `assert`
An `assert` raises `AssertionError` when a claim is false. Writing a few asserts *before* trusting a function is the simplest form of testing.

In [ ]:
def letter_grade(score):
    if not 0 <= score <= 100:
        raise ValueError("score out of range")
    if score >= 70: return "A"
    if score >= 60: return "B"
    if score >= 50: return "C"
    return "F"

assert letter_grade(70) == "A"
assert letter_grade(69) == "B"
assert letter_grade(50) == "C"
assert letter_grade(49) == "F"
try:
    letter_grade(101)
    raise AssertionError("101 should have been rejected")
except ValueError:
    pass
print("All tests passed")

## ✏️ Guided Practice
1. In `safe_divide`, add handling for a `TypeError` (what input causes one?)
2. Add three more `assert` tests to `letter_grade` for boundary values 59, 60 and 100
3. Write `read_positive_int(prompt)` that keeps asking until the user enters an integer greater than 0

## 📝 Class Task
Identify and correct the errors, then classify each as syntax, runtime or logical:
```python
a = int(input("Enter a: "))
b = int(input("Enter b: "))
print(a / b)
```
(Hint: which inputs break it? Make it **error-tolerant**.)

## 🏠 Assignment (To Be Submitted)
Write an **error-tolerant calculator** that:
- Accepts two numbers and an operator (`+ - * /`)
- Rejects invalid numbers and invalid operators with clear messages and asks again
- Handles division by zero
- Includes at least **five `assert` tests** for its calculation function

## 🔎 Reflection Questions
- Why can't Python detect logical errors automatically?
- What is the difference between `else` and `finally`?
- When is it better to *raise* an exception than to return an error message?

---
### ✅ End of Week 13 Lab
Next: **Week 14 – Object-Oriented Programming and Mini Project**